#  Notebook 2: RFM Customer Segmentation

*Recency · Frequency · Monetary scoring  9 actionable segments*

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
import os, warnings
warnings.filterwarnings('ignore')

DB_PATH = os.path.join('..', 'data', 'olist.db')
engine = create_engine(f'sqlite:///{DB_PATH}')


## 1. Compute RFM Metrics

In [2]:
rfm_query = """
SELECT
    c.customer_unique_id,
    MIN(o.customer_id)              AS customer_id,
    MAX(o.order_purchase_timestamp) AS last_purchase,
    COUNT(DISTINCT o.order_id)      AS frequency,
    SUM(op.payment_value)           AS monetary
FROM orders o
JOIN customers c          ON o.customer_id = c.customer_id
JOIN order_payments op    ON o.order_id    = op.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
"""
# Note: Olist uses anonymised per-transaction customer_id values.
# We group by customer_unique_id to track real repeat purchasers.
rfm_raw = pd.read_sql(rfm_query, engine)
rfm_raw['last_purchase'] = pd.to_datetime(rfm_raw['last_purchase'])
reference_date = rfm_raw['last_purchase'].max() + pd.Timedelta(days=1)
rfm_raw['recency'] = (reference_date - rfm_raw['last_purchase']).dt.days
print(f"Reference date : {reference_date.date()}")
print(f"Customers      : {len(rfm_raw):,}")
print(f"Repeat buyers  : {(rfm_raw['frequency'] > 1).sum():,} ({(rfm_raw['frequency'] > 1).mean()*100:.1f}%)")
print(rfm_raw[['customer_id','frequency','monetary','recency']].head())


Reference date : 2018-08-30
Customers      : 93,357
Repeat buyers  : 2,801 (3.0%)
                        customer_id  frequency  monetary  recency
0  fadbb3709178fc513abc1b2670aa1ad2          1    141.90      112
1  4cb282e167ae9234755102258dd52ee8          1     27.19      115
2  9b3932a6253894a02c1df9d19004239f          1     86.22      537
3  914991f0c02ef0843c0e7010c819d642          1     43.62      321
4  47227568b10f5f58a524a75507e6992c          1    196.89      288


## 2. Score R, F, M (1–5 quantile-based)

In [3]:
def score_rfm(df):
    df = df.copy()
    df['R'] = pd.qcut(df['recency'], q=5, labels=[5,4,3,2,1]).astype(int)
    df['F'] = pd.qcut(df['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
    df['M'] = pd.qcut(df['monetary'], q=5, labels=[1,2,3,4,5]).astype(int)
    df['RFM_Score'] = df['R'].astype(str) + df['F'].astype(str) + df['M'].astype(str)
    df['RFM_Total']  = df['R'] + df['F'] + df['M']
    return df

rfm = score_rfm(rfm_raw)
print(rfm[['customer_id','R','F','M','RFM_Score','RFM_Total']].head(10).to_string(index=False))


                     customer_id  R  F  M RFM_Score  RFM_Total
fadbb3709178fc513abc1b2670aa1ad2  4  1  4       414          9
4cb282e167ae9234755102258dd52ee8  4  1  1       411          6
9b3932a6253894a02c1df9d19004239f  1  1  2       112          4
914991f0c02ef0843c0e7010c819d642  2  1  1       211          4
47227568b10f5f58a524a75507e6992c  2  1  4       214          7
4a913a170c26e3c8052ed0202849b5a8  4  1  4       414          9
d2509c13692836fc0449e88cf9eb4858  4  1  1       411          6
a81ebb9b32f102298c0c89635b4b3154  3  1  5       315          9
3b37fb626fdf46cd99d37ec62afa88ff  1  1  4       114          6
c59e8ff99836e90d8b457d4122dc34e9  4  1  3       413          8


## 3. Assign Customer Segments

In [4]:
def assign_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if   r >= 4 and f >= 4 and m >= 4:          return 'Champions'
    elif r >= 3 and f >= 3:                      return 'Loyal Customers'
    elif r >= 4 and f <= 2:                      return 'Recent Customers'
    elif r >= 3 and f <= 2 and m >= 3:           return 'Potential Loyalists'
    elif r == 2 and f >= 3:                      return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:           return 'Cant Lose Them'
    elif r <= 2 and f >= 3:                      return 'Hibernating'
    elif r <= 2 and f <= 2 and m <= 2:           return 'Lost'
    else:                                        return 'Need Attention'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)
print(rfm['Segment'].value_counts().to_string())


Segment
Loyal Customers        27292
Recent Customers       14984
At Risk                11150
Hibernating            11079
Cant Lose Them          8671
Champions               6493
Lost                    6315
Potential Loyalists     4351
Need Attention          3022


## 4. Visualisations

In [5]:
# 3D Scatter
sample = rfm.sample(min(5000, len(rfm)), random_state=42)
fig_3d = px.scatter_3d(sample, x='recency', y='frequency', z='monetary',
    color='Segment', opacity=0.72,
    title=' 3D RFM Customer Segmentation (sample 5k)',
    labels={'recency':'Recency (days)','frequency':'Frequency','monetary':'Monetary (BRL)'},
    color_discrete_sequence=px.colors.qualitative.Bold)
fig_3d.update_layout(template='plotly_dark', font_family='Inter', height=620)
fig_3d.show()

# Donut
fig_donut = px.pie(rfm, names='Segment', hole=0.52,
    title=' Customer Segment Distribution',
    color_discrete_sequence=px.colors.qualitative.Bold)
fig_donut.update_layout(template='plotly_dark', font_family='Inter')
fig_donut.show()


## 5. Segment Profile

In [6]:
seg_profile = rfm.groupby('Segment').agg(
    customers   =('customer_id','count'),
    avg_recency =('recency','mean'),
    avg_freq    =('frequency','mean'),
    avg_monetary=('monetary','mean'),
    total_rev   =('monetary','sum')
).round(2).sort_values('total_rev', ascending=False)

print("\n=== Segment Profile ===")
print(seg_profile.to_string())

fig_bar = px.bar(seg_profile.reset_index(), x='Segment', y='total_rev',
    color='avg_recency', color_continuous_scale='RdYlGn_r',
    title=' Total Revenue by Segment')
fig_bar.update_layout(template='plotly_dark', font_family='Inter')
fig_bar.show()



=== Segment Profile ===
                     customers  avg_recency  avg_freq  avg_monetary   total_rev
Segment                                                                        
Loyal Customers          27292       144.23      1.03        134.02  3657589.17
Recent Customers         14984        90.88      1.00        163.43  2448804.73
Cant Lose Them            8671       394.55      1.00        240.98  2089563.93
Champions                 6493        91.11      1.18        312.13  2026656.84
At Risk                  11150       316.45      1.05        169.40  1888807.04
Hibernating              11079       473.35      1.04        164.54  1822977.13
Potential Loyalists       4351       220.67      1.00        223.25   971340.62
Lost                      6315       396.82      1.00         55.83   352568.09
Need Attention            3022       219.62      1.00         54.32   164154.22


## 6. Export

In [7]:
os.makedirs('../outputs', exist_ok=True)
rfm.to_csv('../outputs/rfm_segments.csv', index=False)
